In [33]:
# TODO: current 4.59 it/s
# TODO: word tokenizer
# TODO: visualize attention
# TODO: visualize gradient issues
# TODO: without bias?
# TODO: make sampling faster
# TODO: how to measure model quality?
# TODO: karpathy tokenizer video
# TODO: add validation set / step
# TODO: load gpt 2 checkpoint 

BASE_CONFIG = {
    "n_positions": 128, # TODO: call block_size
    "batch_size": 2048 *2, #128,,
    "n_embd": 64, #
    "n_head": 2, # TODO: why can't I use 6 heads?
    "n_layer": 2,
    "learning_rate": 3e-3,
    "max_steps": 500,
    #"dropout": 0.1,
    "gradient_accumulation_steps": 1,
    "tune_batch_size": True,
    "layer_norm_enabled": True
}

GPT2_CONFIG = {
        "n_embd": 768,
      "n_layer": 12,        # Currently 6, needs to be 12
      "n_head": 12,
      "n_positions": 1024,  # Currently 256, needs to be 1024
      "vocab_size": 50257,  # tiktoken already gives you this
}

CONFIG = {
    **BASE_CONFIG,
    **GPT2_CONFIG
}


In [34]:
import torch
DEVICE = torch.device("mps")

Load dataset:

In [35]:
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATASET_TEXT = requests.get(url).text
print("n_tokens", len(DATASET_TEXT))

n_tokens 1115394


In [36]:
with open("data/lotr.txt", "r", encoding="latin-1") as f: DATASET_TEXT = f.read()

In [37]:
chars = sorted(list(set(DATASET_TEXT)))
len(chars), chars

(81,
 ['\n',
  ' ',
  '!',
  '"',
  "'",
  '(',
  ')',
  '*',
  ',',
  '-',
  '.',
  '/',
  '0',
  '1',
  '2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9',
  ':',
  ';',
  '=',
  '?',
  'A',
  'B',
  'C',
  'D',
  'E',
  'F',
  'G',
  'H',
  'I',
  'J',
  'K',
  'L',
  'M',
  'N',
  'O',
  'P',
  'Q',
  'R',
  'S',
  'T',
  'U',
  'V',
  'W',
  'X',
  'Y',
  'Z',
  'a',
  'b',
  'c',
  'd',
  'e',
  'f',
  'g',
  'h',
  'i',
  'j',
  'k',
  'l',
  'm',
  'n',
  'o',
  'p',
  'q',
  'r',
  's',
  't',
  'u',
  'v',
  'w',
  'x',
  'y',
  'z',
  '½',
  '¿',
  'ï'])

Create tokenizer:

In [38]:
ctoi = {c:i for i, c in enumerate(chars)}
itoc = {i:c for i, c in enumerate(chars)}
encode = lambda text: [ctoi[c] for c in text]
decode = lambda tokens: "".join([itoc[i] for i in tokens])
decode(encode("hello world")), encode("hello world")
vocab_size = len(ctoi)

In [39]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
CONFIG["vocab_size"] = vocab_size
encode = tokenizer.encode
decode = tokenizer.decode
decode(encode("hello world")), encode("hello world")

('hello world', [31373, 995])

Create token embedding table:

In [ ]:
import torch
import torch.nn as nn

n_embd = CONFIG["n_embd"]
CONFIG["vocab_size"] = vocab_size

PAD_ID = 0
n_positions = CONFIG["n_positions"]
#pad = lambda text: " " * max(0, n_positions - len(text)) + text
#text = pad("hello") # TODO: this is 
tokens = encode("hello") # TODO: not accurate, we should load seq len of tokens and not seqlen of chars
pad = lambda tokens: tokens + [PAD_ID] * (n_positions - len(tokens))
tokens = pad(tokens)
padded = torch.tensor(tokens)
token_embedding_table = nn.Embedding(vocab_size, n_embd)
tokens_emb = token_embedding_table(padded)
tokens_emb.shape, tokens_emb

(torch.Size([1024, 768]),
 tensor([[ 0.7302,  0.0883,  0.5051,  ..., -1.4922, -0.8802,  1.5207],
         [ 0.7302,  0.0883,  0.5051,  ..., -1.4922, -0.8802,  1.5207],
         [ 0.7302,  0.0883,  0.5051,  ..., -1.4922, -0.8802,  1.5207],
         ...,
         [-1.7270, -0.4632,  0.1542,  ...,  0.7199,  0.4024,  0.0665],
         [-1.7270, -0.4632,  0.1542,  ...,  0.7199,  0.4024,  0.0665],
         [-1.7270, -0.4632,  0.1542,  ...,  0.7199,  0.4024,  0.0665]],
        grad_fn=<EmbeddingBackward0>))

Create position embedding table:

In [41]:
n_positions = CONFIG["n_positions"]
n_embd = CONFIG["n_embd"]

position_embedding_table = nn.Embedding(n_positions, n_embd)
positions = torch.arange(n_positions)
positions_emb = position_embedding_table(positions)
positions_emb.shape, positions_emb

(torch.Size([1024, 768]),
 tensor([[ 0.0913,  0.1428, -0.2821,  ..., -0.2345, -1.4707,  0.6293],
         [-0.5376,  1.1795,  2.6485,  ..., -0.7204,  0.2096,  1.1549],
         [-0.8587, -1.4379, -0.2421,  ...,  0.6969, -0.9933,  0.2542],
         ...,
         [-1.0097,  0.3739,  0.3201,  ...,  0.0039, -2.0053, -0.5420],
         [-1.7147,  0.8203,  1.8679,  ..., -1.1147, -0.8692,  0.6938],
         [ 0.6060, -0.8473, -1.8272,  ...,  0.0864, -1.1151, -1.3696]],
        grad_fn=<EmbeddingBackward0>))

Test combining token embeddings and positions embeddings:

In [42]:
x = tokens_emb + positions_emb
x.shape, x

(torch.Size([1024, 768]),
 tensor([[ 0.8214,  0.2311,  0.2230,  ..., -1.7267, -2.3509,  2.1501],
         [ 0.1925,  1.2679,  3.1536,  ..., -2.2126, -0.6706,  2.6756],
         [-0.1285, -1.3496,  0.2630,  ..., -0.7953, -1.8735,  1.7749],
         ...,
         [-2.7366, -0.0893,  0.4744,  ...,  0.7238, -1.6029, -0.4755],
         [-3.4416,  0.3572,  2.0222,  ..., -0.3947, -0.4668,  0.7603],
         [-1.1210, -1.3104, -1.6729,  ...,  0.8063, -0.7127, -1.3031]],
        grad_fn=<AddBackward0>))

Create batch sampler:

In [43]:
DATASET_TOKENS = encode(DATASET_TEXT)

def sample_batch(batch_len):
    n_positions = CONFIG["n_positions"]
    xs, ys = [], []
    for _ in range(batch_len):
        offset_idx = torch.randint(len(DATASET_TOKENS) - n_positions, (1,))
        x = DATASET_TOKENS[offset_idx:offset_idx+n_positions]
        y = DATASET_TOKENS[offset_idx+1:offset_idx+n_positions+1]
        xs.append(x)
        ys.append(y)
    return torch.tensor(xs), torch.tensor(ys)

x, y = sample_batch(10)
x.shape, x, y.shape, y


(torch.Size([10, 1024]),
 tensor([[  514,  9975,   422,  ...,   286,  8465,    12],
         [  257,  2042,  3355,  ...,  1021,   373,   257],
         [   13,   366,  2953,  ...,   307,  1364,   379],
         ...,
         [10578,   286,  1310,  ...,  6290,    12, 45462],
         [  329,  5675,  2474,  ...,   649,  2330,  8946],
         [ 9895,  2474,   531,  ...,    11,   366,  5832]]),
 torch.Size([10, 1024]),
 tensor([[ 9975,   422,   262,  ...,  8465,    12, 49502],
         [ 2042,  3355,   286,  ...,   373,   257, 15061],
         [  366,  2953,   262,  ...,  1364,   379,   262],
         ...,
         [  286,  1310, 41901,  ...,    12, 45462,  6044],
         [ 5675,  2474,   484,  ...,  2330,  8946,   287],
         [ 2474,   531,  9734,  ...,   366,  5832,   750]]))

Test decoding batch:

In [44]:
for i in range(x.shape[0]):
    x_tokens = x[i].tolist()
    y_tokens = y[i].tolist()
    print(f"x {i}: {decode(x_tokens)}")
    print(f"y {i}: {decode(y_tokens)}")
    print()

x 0:  us tonight from the peril that comes behind."
"If Elves indeed still dwell here in the darkening world," said Gimli.
"It is long since any of my own folk journeyed hither back to the land whence we wandered in ages long ago," said Legolas, "but we hear that Lï¿½rien is not yet deserted, for there is a secret power here that holds evil from the land. Nevertheless its folk are seldom seen, and maybe they dwell now deep in the woods and far from the northern border."
"Indeed deep in the wood they dwell," said Aragorn, and sighed as if some memory stirred in him. "We must fend for ourselves tonight. We will go forward a short way, until the trees are all about us, and then we will turn aside from the path and seek a place to rest in."
He stepped forward; but Boromir stood irresolute and did not follow. "Is there no other way? " he said.
"What other fairer way would you desire? " said Aragorn.
"A plain road, though it led through a hedge of swords," said Boromir. "By strange paths has

In [ ]:
batch_size = CONFIG["batch_size"]
n_positions = CONFIG["n_positions"]
x, _ = sample_batch(batch_size)
x_emb = token_embedding_table(x)
positions = torch.arange(n_positions)
pos_emb = position_embedding_table(positions)
x = x_emb + pos_emb
x.shape, x[0][0]

In [ ]:
qkv_shape = (CONFIG["n_embd"], CONFIG["n_embd"]) # TODO; call embed_dim
Wq = torch.randn(qkv_shape)
Wk = torch.randn(qkv_shape)
Wv = torch.randn(qkv_shape)

In [ ]:
Q = x @ Wq
K = x @ Wk
V = x @ Wv
Q.shape, K.shape, V.shape

In [ ]:
Kt = K.transpose(-2, -1)
Kt.shape

In [ ]:
QKt = Q @ Kt # (B, T, C) @ (B, C, T) -> (B, T, T)
QKt.shape, QKt[0][0]

In [ ]:
QKt_scaled = QKt / torch.sqrt(torch.tensor(n_embd))
QKt_scaled.shape, QKt_scaled[0][0]

In [ ]:
tril = torch.tril(torch.ones((n_positions, n_positions)))
tril

In [ ]:
mask = tril == 0
mask

In [ ]:
QKt_scaled_masked = QKt_scaled.masked_fill(mask, float("-inf"))
QKt_scaled_masked.shape, QKt_scaled_masked[0]

In [ ]:
from torch.nn import functional as F
import matplotlib.pyplot as plt

attention = F.softmax(QKt_scaled_masked, dim=-1)

plt.imshow(attention[0].detach().cpu(), cmap='viridis')
plt.colorbar()
plt.title("Attention Map")
plt.xlabel("Key positions")
plt.ylabel("Query positions")
plt.show()

In [ ]:
output = attention @ V
output.shape

In [ ]:
ffn1 = nn.Linear(n_embd, n_embd * 4)
gelu = nn.GELU()
ffn2 = nn.Linear(n_embd * 4, n_embd)
n_heads = CONFIG["n_head"]
head_size = n_embd // n_heads
lm_head = nn.Linear(n_embd, head_size)
output_head = lm_head(ffn2(gelu(ffn1(output))))
output_head.shape, output_head

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, head_size):
        nn.Module.__init__(self)
        
        self.head_size = head_size
        n_embd = CONFIG["n_embd"]

        #self.W_q = nn.Linear(n_embd, head_size)
        #self.W_k = nn.Linear(n_embd, head_size)
        #self.W_v = nn.Linear(n_embd, head_size)

        self.W_qkv = nn.Linear(n_embd, head_size * 3)

        n_positions = CONFIG["n_positions"]
        self.register_buffer("tril", torch.tril(torch.ones(n_positions, n_positions)))
        
    def forward(self, x):
        B, T, C = x.shape
        #print("WORKED BTC", B, T, C)

        #Q = self.W_q(x)
        #K = self.W_k(x)
        #V = self.W_v(x)

        QKV = self.W_qkv(x)
        Q, K, V = torch.chunk(QKV, 3, dim=-1)
        
        Kt = K.transpose(-2, -1)
        QKt = Q @ Kt
        QKt_scaled = QKt / torch.sqrt(torch.tensor(self.head_size))
        #mask = self.tril == 0

        mask = self.tril[:T, :T].to(x.device).bool()   # (T, T)
        #mask = mask.unsqueeze(0)   # (1, 1, T, T) if needed
        #print("MASK", mask.shape)
        #print("QKt_scaled", QKt_scaled.shape)

        QKt_masked = QKt_scaled.masked_fill(~mask, float("-inf"))
        attention = F.softmax(QKt_masked, dim=-1) # TODO: confirm this
        out = attention @ V

        return out

x, y = sample_batch(1)
x = token_embedding_table(x)
out = AttentionHead(3)(x)
out.shape, out

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)
        
        n_heads = CONFIG["n_head"]
        n_embd = CONFIG["n_embd"]
        head_size = n_embd // n_heads
        self.heads = nn.ModuleList([AttentionHead(head_size) for _ in range(n_heads)])
        self.W_o = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        outs = []
        for head in self.heads: # TODO: for here?
            out = head(x)
            outs.append(out)
        mha_out = torch.concat(outs, dim=-1)
        mha_out = self.W_o(mha_out)
        return mha_out

x, _ = sample_batch(1)
x = token_embedding_table(x)
out = MultiHeadAttention()(x)
out.shape, out

In [ ]:
a = torch.randn((3, 3))
a

In [ ]:
tril = torch.tril(torch.ones(3, 3))
tril

In [ ]:
mask = tril == 0
mask

In [ ]:
a.masked_fill(mask, float("-inf"))

In [ ]:
class FastMultiHeadAttention(nn.Module):
    def __init__(self, n_heads):
        nn.Module.__init__(self)
        n_embd = CONFIG["n_embd"]
        block_size = CONFIG["n_positions"]
        head_size = n_embd // n_heads
        self.n_heads = n_heads
        self.head_size = head_size
        self.W_qkv = nn.Linear(n_embd, n_embd * 3)
        self.W_o = nn.Linear(n_embd, n_embd)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        QKV = self.W_qkv(x) # (B, T, 3 * n_embd)
        Q, K, V = torch.chunk(QKV, 3, dim=-1)
        Q = Q.view(B, T, self.n_heads, self.head_size).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_size).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_size).transpose(1, 2)
        Kt = K.transpose(-2, -1)
        QKt = Q @ Kt
        QKt_scaled = QKt / torch.sqrt(torch.tensor(self.head_size)).to(x.device)
        mask = self.tril[:T, :T].bool().unsqueeze(0).unsqueeze(0)
        QKt_masked = QKt_scaled.masked_fill(~mask, float("-inf"))
        attention = F.softmax(QKt_masked, dim=-1)
        V_scaled = attention @ V # (B, n_heads, T, head_size)
        V_scaled_t = V_scaled
        V_scaled_t_realigned = V_scaled_t.transpose(1, 2).contiguous().view(B, T, C)
        output = self.W_o(V_scaled_t_realigned) 
        return output

x, _ = sample_batch(1)
x = token_embedding_table(x)
out = FastMultiHeadAttention(2)(x)
out.shape, out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)
        
        n_embd = CONFIG["n_embd"]
        self.ffn1 = nn.Linear(n_embd, n_embd * 4)
        self.gelu = nn.GELU(approximate='tanh')  # TODO: hugging face does this... why?
        self.ffn2 = nn.Linear(n_embd * 4, n_embd)

    def forward(self, x):
        x = self.ffn1(x)
        x = self.gelu(x)
        x = self.ffn2(x)
        return x

x,y = sample_batch(1)
x = token_embedding_table(x)
mha_out = MultiHeadAttention()(x)
out = FeedForward()(mha_out)
out.shape, out

In [ ]:
class Block(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)
        self.head = FastMultiHeadAttention(CONFIG["n_head"])
        self.ffn = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        if CONFIG["layer_norm_enabled"]:
            x = x + self.head(self.ln1(x))  # Normalize input to attention
        else:
            x = x + self.head(x)
        
        if CONFIG["layer_norm_enabled"]:
            x = x + self.ffn(self.ln2(x))  # Normalize input to FFN
        else:
            x = x + self.ffn(x)
        return x

x, y = sample_batch(1)
x = token_embedding_table(x)
out = Block()(x)
out.shape, out

In [ ]:
tokens = encode(pad("hello"))
tokens_t = torch.tensor(tokens)
tokens_t = tokens_t.view(1, n_positions)
tokens_t

In [ ]:
class Transformer(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(CONFIG["n_positions"], n_embd)

        # TODO: seq vs parallel
        self.blocks = nn.ModuleList([Block() for _ in range(CONFIG["n_layer"])])


        self.lm_head = nn.Linear(CONFIG["n_embd"], CONFIG["vocab_size"])
        
        # TODO: hugging face does this... why?
        self.lm_head.weight = self.token_embedding_table.weight  # Share weights!

        self.ln_final = nn.LayerNorm(n_embd)

    def forward(self, x):
        B, T = x.shape
        tok_emb = self.token_embedding_table(x)        # (B, T, C)
        pos = torch.arange(T, device=x.device)         # (T,)
        pos_emb = self.position_embedding_table(pos)   # (T, C)
        pos_emb = pos_emb.unsqueeze(0)                 # (1, T, C) -> broadcasts over batch
        x = tok_emb + pos_emb      
        for block in self.blocks: x = block(x)
        if CONFIG["layer_norm_enabled"]: x = self.ln_final(x)
        x = self.lm_head(x)
        return x

    @torch.no_grad()
    def generate(self, text, max_len=20):
        device = next(self.parameters()).device
        n_positions = CONFIG["n_positions"]
        tokens = encode(text)
        tokens_t = torch.tensor(tokens).to(device)
        tokens_t = tokens_t.unsqueeze(0)#view(1, -1)
        output_tokens = []
        while len(output_tokens) < max_len:
            logits = self(tokens_t)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx = torch.multinomial(probs, num_samples=1)
            idx = idx.squeeze().item()
            tokens.append(idx)
            tokens = tokens[-n_positions:]
            tokens_t = torch.tensor(tokens).to(device).unsqueeze(0)
            #tokens_t = tokens_t.view(1, n_positions)
            output_tokens.append(idx)
        text = decode(output_tokens)
        return text


x, y = sample_batch(1)
out = Transformer()(x)
out.shape, out[0][0]

#text = Transformer().generate("hello")
#print(text)

In [ ]:
from torch.utils.data import Dataset

class TrainDataset(Dataset):
    def __init__(self, tokens, n_positions):
        self.tokens = tokens
        self.n_positions = n_positions

    def __getitem__(self, idx):
        x = self.tokens[idx:idx+self.n_positions]
        y = self.tokens[idx+1:idx+self.n_positions+1]
        return torch.tensor(x), torch.tensor(y)

    def __len__(self):
        return len(self.tokens) - self.n_positions

tokens = encode(DATASET_TEXT)
train_dataset = TrainDataset(tokens, CONFIG["n_positions"])
x, y = next(iter(train_dataset))
print("x:" + decode(x.tolist())[:10])
print("y:" + decode(y.tolist())[:10])
#print(decode(first_seq))

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=0,  # MPS doesn't benefit from >0 workers
    pin_memory=False,  # Not needed for MPS
    drop_last=True  # Ensures consistent batch sizes
)
for x, y in train_loader:
    for i in range(10):
        print("x:" + decode(x[i].tolist())[:10])
        print("y:" + decode(y[i].tolist())[:10])
        print("---")
    break

In [ ]:
model = Transformer()
#model = torch.compile(model, mode="reduce-overhead") # TODO: no gain on mps?
model.to(DEVICE)
model.train()

In [ ]:
if CONFIG["tune_batch_size"]:
    from aiml_notebooks import find_optimal_batch_config
    result = find_optimal_batch_config(
        model=model,
        vocab_size=CONFIG["vocab_size"],
        n_positions=CONFIG["n_positions"],
    )
    recommended = result['recommended']
    if CONFIG["batch_size"] == "auto": CONFIG["batch_size"] = recommended['batch_size']
    if CONFIG["gradient_accumulation_steps"] == "auto": CONFIG["gradient_accumulation_steps"] = recommended['gradient_accumulation_steps']

In [ ]:
# Add this to your training loop to monitor gradients
def get_grad_norm(model):
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    total_norm = total_norm ** (1. / 2)
    return total_norm

# In your training loop, after loss.backward():
grad_norm = get_grad_norm(model)
grad_norm

In [ ]:
from tqdm import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

num_epochs = 100
losses = []
grad_norms = []

gradient_accumulation_steps = CONFIG["gradient_accumulation_steps"]
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch_idx, (x, y) in enumerate(batch_bar):
        x, y = x.to(DEVICE), y.to(DEVICE)
        
        logits = model(x)
        logits = logits.permute(0, 2, 1)
        loss = F.cross_entropy(logits, y)
    
        losses.append(loss.item())

        (loss / gradient_accumulation_steps).backward()

        if ((batch_idx + 1) % gradient_accumulation_steps == 0) or (batch_idx + 1 == len(train_loader)):
            grad_norm = get_grad_norm(model)  # compute *once* per update
            grad_norms.append(grad_norm)

            batch_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                grad_norm=f"{grad_norm:.4f}"
            )

            optimizer.step()
            optimizer.zero_grad()
        else:
            batch_bar.set_postfix(loss=f"{loss.item():.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 1 row, 2 columns

# Left plot
axes[0].plot(losses)
axes[0].set_title("Losses")

# Right plot
axes[1].plot(grad_norms)   # replace with your second dataset
axes[1].set_title("Grad Norms")

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
print(model.generate("Frodo picked up the sword", max_len=1024))

In [ ]:
# Track activations - respect the layer_norm_enabled flag
def track_block_activations(model, x):
    """Track activations at the output of each block."""
    stats = []
    
    model.eval()
    with torch.no_grad():
        # Get embeddings
        x = model.token_embedding_table(x)
        pos = torch.arange(x.size(1), device=x.device)
        pos_emb = model.position_embedding_table(pos).unsqueeze(0)
        x = x + pos_emb
        
        # Track initial state
        stats.append({
            'layer': 'input',
            'mean': x.mean().item(),
            'std': x.std().item(),
            'max_abs': x.abs().max().item()
        })
        
        # Track through each block
        for i, block in enumerate(model.blocks):
            x = block(x)
            stats.append({
                'layer': f'block_{i}',
                'mean': x.mean().item(),
                'std': x.std().item(),
                'max_abs': x.abs().max().item()
            })
        
        # Only apply final layer norm if enabled (same as model's forward pass)
        if CONFIG["layer_norm_enabled"]:
            x = model.ln_final(x)
            stats.append({
                'layer': 'final_ln',
                'mean': x.mean().item(),
                'std': x.std().item(),
                'max_abs': x.abs().max().item()
            })
        else:
            # Track the raw output from last block (no normalization)
            stats.append({
                'layer': 'final',
                'mean': x.mean().item(),
                'std': x.std().item(),
                'max_abs': x.abs().max().item()
            })
    
    return stats

# Get a sample batch
x, y = sample_batch(1)
x = x.to(DEVICE)

# Track activations
stats = track_block_activations(model, x)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

layers = [s['layer'] for s in stats]
max_abs = [s['max_abs'] for s in stats]
stds = [s['std'] for s in stats]

# Plot 1: Maximum absolute values (log scale)
axes[0].semilogy(max_abs, marker='o', linewidth=2, markersize=8, color='red')
axes[0].set_title('Maximum Absolute Activation Values', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Layer', fontsize=12)
axes[0].set_ylabel('Max Abs Value (log scale)', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(len(layers)))
axes[0].set_xticklabels(layers, rotation=45, ha='right')

# Plot 2: Standard deviation
axes[1].plot(stds, marker='s', linewidth=2, markersize=8, color='blue')
axes[1].set_title('Standard Deviation of Activations', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Layer', fontsize=12)
axes[1].set_ylabel('Standard Deviation', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(range(len(layers)))
axes[1].set_xticklabels(layers, rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*70)
print("ACTIVATION STATISTICS")
print("="*70)
print(f"{'Layer':<15} {'Mean':>12} {'Std':>12} {'Max Abs':>12}")
print("-"*70)
for s in stats:
    print(f"{s['layer']:<15} {s['mean']:>12.4f} {s['std']:>12.4f} {s['max_abs']:>12.4f}")
print("="*70)

# Calculate growth rate
if len(max_abs) > 1:
    growth_rate = max_abs[-1] / max_abs[0] if max_abs[0] != 0 else float('inf')
    print(f"\nActivation Growth Rate (Final/Initial): {growth_rate:.2f}x")
    print(f"Initial max abs: {max_abs[0]:.4f}")
    print(f"Final max abs: {max_abs[-1]:.4f}")

In [ ]:
def load_pretrained_weights(model, model_name='openai-community/gpt2'):
    """
    Load pretrained weights from HuggingFace into our custom GPT-2 model.
    
    Args:
        model: Our custom Transformer model instance
        model_name: HuggingFace model identifier (default: 'openai-community/gpt2')
    
    Returns:
        model: The model with loaded weights
    """
    from transformers import GPT2LMHeadModel
    
    print(f"Loading pretrained weights from {model_name}...")
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    hf_state_dict = hf_model.state_dict()
    
    # Create mapping from HuggingFace names to our names
    our_state_dict = model.state_dict()
    
    # 1. Token and position embeddings
    our_state_dict['token_embedding_table.weight'].copy_(hf_state_dict['transformer.wte.weight'])
    our_state_dict['position_embedding_table.weight'].copy_(hf_state_dict['transformer.wpe.weight'])
    
    # 2. Final layer norm
    our_state_dict['ln_final.weight'].copy_(hf_state_dict['transformer.ln_f.weight'])
    our_state_dict['ln_final.bias'].copy_(hf_state_dict['transformer.ln_f.bias'])
    
    # 3. Load weights for each transformer block
    n_layers = CONFIG["n_layer"]
    for i in range(n_layers):
        # Layer norm 1 (pre-attention)
        our_state_dict[f'blocks.{i}.ln1.weight'].copy_(hf_state_dict[f'transformer.h.{i}.ln_1.weight'])
        our_state_dict[f'blocks.{i}.ln1.bias'].copy_(hf_state_dict[f'transformer.h.{i}.ln_1.bias'])
        
        # Attention QKV projection (combined)
        # HuggingFace stores as (768, 2304) but PyTorch Linear expects (2304, 768) for weight
        # So we need to transpose
        hf_qkv_weight = hf_state_dict[f'transformer.h.{i}.attn.c_attn.weight']
        hf_qkv_bias = hf_state_dict[f'transformer.h.{i}.attn.c_attn.bias']
        our_state_dict[f'blocks.{i}.head.W_qkv.weight'].copy_(hf_qkv_weight.T)  # Transpose!
        our_state_dict[f'blocks.{i}.head.W_qkv.bias'].copy_(hf_qkv_bias)
        
        # Attention output projection
        hf_o_weight = hf_state_dict[f'transformer.h.{i}.attn.c_proj.weight']
        hf_o_bias = hf_state_dict[f'transformer.h.{i}.attn.c_proj.bias']
        our_state_dict[f'blocks.{i}.head.W_o.weight'].copy_(hf_o_weight.T)  # Transpose!
        our_state_dict[f'blocks.{i}.head.W_o.bias'].copy_(hf_o_bias)
        
        # Layer norm 2 (pre-FFN)
        our_state_dict[f'blocks.{i}.ln2.weight'].copy_(hf_state_dict[f'transformer.h.{i}.ln_2.weight'])
        our_state_dict[f'blocks.{i}.ln2.bias'].copy_(hf_state_dict[f'transformer.h.{i}.ln_2.bias'])
        
        # FFN first layer (expansion)
        hf_fc_weight = hf_state_dict[f'transformer.h.{i}.mlp.c_fc.weight']
        hf_fc_bias = hf_state_dict[f'transformer.h.{i}.mlp.c_fc.bias']
        our_state_dict[f'blocks.{i}.ffn.ffn1.weight'].copy_(hf_fc_weight.T)  # Transpose!
        our_state_dict[f'blocks.{i}.ffn.ffn1.bias'].copy_(hf_fc_bias)
        
        # FFN second layer (projection)
        hf_proj_weight = hf_state_dict[f'transformer.h.{i}.mlp.c_proj.weight']
        hf_proj_bias = hf_state_dict[f'transformer.h.{i}.mlp.c_proj.bias']
        our_state_dict[f'blocks.{i}.ffn.ffn2.weight'].copy_(hf_proj_weight.T)  # Transpose!
        our_state_dict[f'blocks.{i}.ffn.ffn2.bias'].copy_(hf_proj_bias)
    
    # Note: lm_head weights are tied to token_embedding_table.weight, so no separate loading needed
    
    print(f"✓ Successfully loaded {n_layers} transformer blocks")
    print("✓ Weight tying preserved (lm_head shares weights with token embeddings)")
    
    return model

# Load pretrained GPT-2 weights from HuggingFace
model_pretrained = Transformer().to(DEVICE)
model_pretrained = load_pretrained_weights(model_pretrained)

# Test generation with pretrained model
model_pretrained.eval()
print(model_pretrained.generate("Frodo picked up the sword", max_len=1024))